# NYC 9/11 Records Analysis — Data Cleaning Pipeline

This notebook documents and reproduces the data cleaning pipeline implemented in `src/processing/clean.py`. It transforms scraped Parquet tables from `data/final/` into analysis-ready tables in `data/processed/` using pure functions (`DataFrame -> DataFrame`) without modifying the raw source files.

### Pipeline Summary
1. **Off-Topic Filtering (`is_off_topic`)**: Excludes documents in `settings.OFF_TOPIC_BOXES` unless their title or description references WTC/9/11, saving removed records to `removed_documents.parquet`.
2. **`documents` (`clean_documents`)**: Drops empty/constant columns, converts file size strings to numeric MB, parses `pdf_links` JSON arrays, validates dates, applies document date corrections (`dates.py`), and adds `in_study_window`.
3. **`pages` (`clean_pages`)**: Extracts `litigation_stamp` and normalized `litigation_source` (`21MC100` vs. subpoena names), strips portal headers, Bates stamps, and litigation stamps from text, rejoins hyphenated line breaks, computes text-quality metrics (`has_usable_text`), and propagates corrected document dates.
4. **`buildings` (`clean_buildings`) & `building_mentions`**: Parses unit counts, square footage, and building class codes, cleans dates, and flags buildings inside the Lower Manhattan bounding box.
5. **`entities` (`clean_entities`) & `entity_mentions` (`clean_entity_mentions`)**: Reclassifies misfiled public agencies, maps agency slugs to canonical names and levels (`agency_map.py`), normalizes contractor/lab names by stripping corporate suffixes, and attaches canonical `entity_id`s to mentions.
6. **`catalog_changes` (`clean_catalog_changes`)**: Parses and validates `capture_date`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Ensure repository root is on sys.path when running from notebooks/
repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists() and (repo_root.parent / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import config
from src.processing import settings
from src.processing.clean import (
    SOURCE,
    OUT,
    is_off_topic,
    clean_documents,
    clean_pages,
    clean_buildings,
    clean_entities,
    clean_entity_mentions,
    clean_catalog_changes,
    run,
)

print(f'Source directory: {SOURCE}')
print(f'Output directory: {OUT}')

## 1. Load Raw Tables & Filter Off-Topic Documents
Documents located in `settings.OFF_TOPIC_BOXES` are excluded unless their `title` or `description` matches WTC/9/11 keywords (`world trade`, `wtc`, `ground zero`, `9/11`, `september 11`). Removed records are preserved in `removed_documents.parquet` and filtered out of all `doc_id`-keyed tables (`documents`, `pages`, `building_mentions`, `entity_mentions`).

In [ ]:
def load_raw(name: str) -> pd.DataFrame:
    return pd.read_parquet(SOURCE / f'{name}.parquet')

raw_documents = load_raw('documents')
off_topic_mask = is_off_topic(raw_documents)
removed_documents = raw_documents.loc[off_topic_mask, ['doc_id', 'box', 'folder', 'site_date', 'title']]

def filter_on_topic(df: pd.DataFrame) -> pd.DataFrame:
    return df[~df['doc_id'].isin(removed_documents['doc_id'])]

print(f'Total raw documents: {len(raw_documents):,}')
print(f'Off-topic documents removed: {len(removed_documents):,}')
print(f'Retained documents: {(~off_topic_mask).sum():,}')

## 2. Clean `documents`
- Drops empty or constant metadata columns (`title_confidence`, `doc_type`, `doc_type_confidence`, `removed`, `removed_date`).
- Casts `n_pages` to nullable `Int64` and parses numeric `file_size_mb` from strings like `'9 pages / 8.2 MB'`.
- Converts `pdf_links` from JSON-encoded strings into deduplicated Python lists.
- Validates `site_date` (`>= settings.EARLIEST_VALID_DATE`), joins corrected `doc_date` from `dates.py`, and flags `in_study_window`.

In [ ]:
documents = clean_documents(raw_documents[~off_topic_mask])
documents[['doc_id', 'box', 'site_date', 'doc_date', 'in_study_window', 'n_pages', 'file_size_mb']].head()

## 3. Clean `pages` & Compute Text Quality Metrics
- Drops always-empty `entity_links`.
- Extracts `litigation_stamp` and normalized `litigation_source` (`21MC100` vs. OCR-deduplicated subpoena names) before modifying text.
- Strips `NYC 9/11 Public Portal Document` headers, Bates stamps (`NYC-WTC_#########`), and litigation stamps from `text`.
- Collapses redundant whitespace and rejoins words hyphenated across line breaks (`carry-\nout` -> `carryout`).
- Computes `n_chars`, `alpha_ratio`, `word_ratio`, and `has_usable_text` to identify prose-like pages suitable for NLP.
- Maps corrected `doc_date` from `documents` onto each page and sets `date_source` (`document` vs. `none`) and `in_study_window`.

In [ ]:
raw_pages = filter_on_topic(load_raw('pages'))
doc_date_map = documents.set_index('doc_id')['doc_date']
pages = clean_pages(raw_pages, doc_date_map)

print(f'Total retained pages: {len(pages):,}')
print(f'Pages with usable text: {pages["has_usable_text"].sum():,} ({pages["has_usable_text"].mean():.1%})')
pages[['doc_id', 'page_num', 'date', 'litigation_source', 'n_chars', 'alpha_ratio', 'word_ratio', 'has_usable_text']].head()

## 4. Clean `buildings` & `building_mentions`
- Parses `residential_units` and `total_units` from `'X residential / Y total'` strings.
- Extracts numeric `building_area_sqft` and backfills missing `building_class_code` values from `building_class` labels.
- Standardizes `year_built` (replacing `<= 0` with `NA`), `floors`, `first_date`, and `last_date`.
- Flags properties located within `config.LOWER_MANHATTAN_BBOX` (`in_lower_manhattan`).
- Filters `building_mentions` to exclude off-topic documents.

In [ ]:
buildings = clean_buildings(load_raw('buildings'))
building_mentions = filter_on_topic(load_raw('building_mentions'))

print(f'Buildings in Lower Manhattan: {buildings["in_lower_manhattan"].sum():,} / {len(buildings):,}')
buildings[['bin', 'address', 'residential_units', 'total_units', 'building_area_sqft', 'building_class_code', 'in_lower_manhattan']].head()

## 5. Canonicalize `entities` & `entity_mentions`
- Preserves `source_type` as the original join key and reclassifies public agencies misfiled as `contractor` (`CONTRACTOR_SLUGS_THAT_ARE_AGENCIES`) to `agency`.
- Maps agency slugs to canonical `entity_id`, `entity_name`, and `agency_level` via `agency_map.py` (raising a `ValueError` if any unmapped agency slug is encountered).
- Normalizes `contractor` and `lab` names by lowercasing, replacing `&` with `and`, stripping punctuation and corporate suffixes (`Inc.`, `LLC`, `Corp.`, etc.), and assigning the most-mentioned variant as `entity_name` (unless the normalized string is a single generic word).
- Joins `entity_mentions` to canonical `entity_id` and corrected `entity_type`, dropping the constant `confidence` column.

In [ ]:
entities = clean_entities(load_raw('entities'))
entity_mentions = clean_entity_mentions(filter_on_topic(load_raw('entity_mentions')), entities)

print('Entity counts by corrected type:')
print(entities['entity_type'].value_counts())
entities[['source_type', 'slug', 'entity_type', 'entity_id', 'entity_name', 'agency_level', 'n_pages']].head(10)

## 6. Clean `catalog_changes` & Export to `data/processed/`
Parses `capture_date` in `catalog_changes` and writes all 8 cleaned Parquet tables to `data/processed/`.

In [ ]:
catalog_changes = clean_catalog_changes(load_raw('catalog_changes'))

tables = {
    'documents': documents,
    'pages': pages,
    'buildings': buildings,
    'building_mentions': building_mentions,
    'entities': entities,
    'entity_mentions': entity_mentions,
    'catalog_changes': catalog_changes,
    'removed_documents': removed_documents,
}

OUT.mkdir(parents=True, exist_ok=True)
for name, table in tables.items():
    out_path = OUT / f'{name}.parquet'
    table.to_parquet(out_path, index=False)
    print(f'{name:18s} {len(table):>8,} rows -> {out_path}')